# 04 OpenAI Embedding Experiments


In [1]:
from __future__ import annotations

import html
import importlib.util
import json
import math
import os
import re
import time
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import pandas as pd

CODE_DIR = Path('/Users/tiencains/Documents/dissertation/dissertation process/code/phishing_email_detection')
PROJECT_ROOT = Path('/Users/tiencains/Documents/dissertation/dissertation process')
DATA_DIR = PROJECT_ROOT / 'data choosing list' / 'Phishing Email Curated Datasets'
CEAS_PATH = DATA_DIR / 'CEAS_08.csv'
ENRON_PATH = DATA_DIR / 'Enron.csv'
API_KEY_FILE = CODE_DIR / 'openai_api_key.txt'

OUTPUT_DIR = CODE_DIR / 'output'
DATA_OUTPUT_DIR = OUTPUT_DIR / 'data'
STATS_OUTPUT_DIR = OUTPUT_DIR / 'statistics'
FIGURE_OUTPUT_DIR = OUTPUT_DIR / 'figures'
MODEL_OUTPUT_DIR = OUTPUT_DIR / 'models'
EMBEDDING_OUTPUT_DIR = OUTPUT_DIR / 'embeddings'
ERROR_OUTPUT_DIR = OUTPUT_DIR / 'error_analysis'
REPORT_OUTPUT_DIR = OUTPUT_DIR / 'reports'

RANDOM_SEED = 42
EXPECTED_RAW_ROWS = 68921
EXPECTED_RAW_LABEL_COUNTS = {1: 35818, 0: 33103}
SCALABILITY_EMAIL_COUNTS = [10_000, 100_000, 1_000_000]

# I keep pricing assumptions here so I can update the cost section before final writing.
EMBEDDING_PRICES_USD_PER_1M_TOKENS = {
    'text-embedding-3-small': 0.02,
    'text-embedding-3-large': 0.13,
}

API_KEY_EMPTY_MARKERS = {
    '',
    'PENGFEI_OPENAI_KEY_FOR_EMBEDDINGS',
}

os.chdir(CODE_DIR)
os.environ.setdefault('MPLCONFIGDIR', str(OUTPUT_DIR / 'matplotlib_cache'))


def ensure_output_dirs() -> None:
    for path in [
        OUTPUT_DIR,
        DATA_OUTPUT_DIR,
        STATS_OUTPUT_DIR,
        FIGURE_OUTPUT_DIR,
        MODEL_OUTPUT_DIR,
        EMBEDDING_OUTPUT_DIR,
        ERROR_OUTPUT_DIR,
        REPORT_OUTPUT_DIR,
    ]:
        path.mkdir(parents=True, exist_ok=True)


def check_packages(packages: Iterable[str]) -> list[str]:
    return [package for package in packages if importlib.util.find_spec(package) is None]


def require_packages(packages: Iterable[str]) -> None:
    missing = check_packages(packages)
    if missing:
        raise RuntimeError(
            'Missing packages: ' + ', '.join(missing) +
            '. In PyCharm, select the project .venv interpreter, or install these packages in that interpreter.'
        )


def now_seconds() -> float:
    return time.perf_counter()


def elapsed(start: float) -> float:
    return round(time.perf_counter() - start, 4)


def save_json(data: dict[str, Any], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding='utf-8')


def load_openai_api_key() -> str | None:
    env_key = os.getenv('OPENAI_API_KEY', '').strip()
    if env_key and env_key not in API_KEY_EMPTY_MARKERS:
        return env_key
    if not API_KEY_FILE.exists():
        return None
    for raw_line in API_KEY_FILE.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#'):
            continue
        if line.startswith('OPENAI_API_KEY='):
            line = line.split('=', 1)[1].strip()
        line = line.strip().strip('"').strip("'")
        if line and line not in API_KEY_EMPTY_MARKERS:
            os.environ['OPENAI_API_KEY'] = line
            return line
    return None


def standardize_email_frame(df: pd.DataFrame, source_name: str) -> pd.DataFrame:
    df = df.copy()
    df['source_dataset'] = source_name
    for col in ['sender', 'receiver', 'date', 'subject', 'body', 'label', 'urls']:
        if col not in df.columns:
            df[col] = pd.NA
    df['subject'] = df['subject'].fillna('').astype(str)
    df['body'] = df['body'].fillna('').astype(str)
    df['text'] = (
        df['subject'].str.cat(df['body'], sep=' ')
        .str.replace(r'\s+', ' ', regex=True)
        .str.strip()
    )
    df['label'] = pd.to_numeric(df['label'], errors='coerce').astype('Int64')
    return df[['source_dataset', 'sender', 'receiver', 'date', 'subject', 'body', 'text', 'label', 'urls']]


URL_RE = re.compile(r'(https?://\S+|www\.\S+)', flags=re.IGNORECASE)
EMAIL_RE = re.compile(r'\b[\w.\-+%]+@[\w.\-]+\.[A-Za-z]{2,}\b')
HTML_TAG_RE = re.compile(r'<[^>]+>')
CONTROL_RE = re.compile(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]')
NUMBER_RE = re.compile(r'\b\d+(?:[.,:/-]\d+)*\b')
MULTISPACE_RE = re.compile(r'\s+')


def strip_html(text: str) -> str:
    text = html.unescape(str(text))
    return HTML_TAG_RE.sub(' ', text)


def clean_text_level1(text: str) -> str:
    text = strip_html(text)
    text = CONTROL_RE.sub(' ', text)
    text = URL_RE.sub(' URLTOKEN ', text)
    text = EMAIL_RE.sub(' EMAILTOKEN ', text)
    text = NUMBER_RE.sub(' NUMBERTOKEN ', text)
    text = text.lower()
    text = re.sub(r"[^a-z0-9_<>@\s\-']", ' ', text)
    return MULTISPACE_RE.sub(' ', text).strip()


def make_dedup_key(text: pd.Series) -> pd.Series:
    return (
        text.fillna('').astype(str).map(strip_html).str.lower()
        .str.replace(r'\s+', ' ', regex=True).str.strip()
    )


def simple_tokenize(text: str) -> list[str]:
    return re.findall(r"[a-z0-9_<>@\-']+", str(text).lower())


OBFUSCATION_MAP = {
    'password': 'p a s s w 0 r d',
    'account': 'acc0unt',
    'verify': 'ver!fy',
    'login': 'log-in',
    'click': 'cl1ck',
    'urgent': 'urg3nt',
    'security': 'sec urity',
    'bank': 'b@nk',
    'payment': 'pay ment',
    'update': 'up date',
}


def perturb_text_level2(text: str) -> str:
    text = strip_html(text)
    text = URL_RE.sub(' hxxps://example[.]com/secure-login ', text)
    text = EMAIL_RE.sub(' support [at] example [dot] com ', text)
    pattern = re.compile(r'\b(' + '|'.join(map(re.escape, OBFUSCATION_MAP)) + r')\b', re.I)

    def replace_word(match: re.Match[str]) -> str:
        word = match.group(0)
        replacement = OBFUSCATION_MAP.get(word.lower(), word)
        return replacement.upper() if word.isupper() else replacement

    text = pattern.sub(replace_word, text)
    text = re.sub(r'([!?.,;:])', r' \1 ', text)
    return re.sub(r'\s+', ' ', text).strip()


def add_text_length_columns(df: pd.DataFrame, column: str = 'text') -> pd.DataFrame:
    df = df.copy()
    df['char_count'] = df[column].fillna('').astype(str).str.len()
    df['word_count'] = df[column].fillna('').astype(str).str.split().str.len()
    return df


def validate_raw_counts(df: pd.DataFrame) -> None:
    actual_rows = len(df)
    actual_counts = {int(k): int(v) for k, v in df['label'].value_counts(dropna=False).to_dict().items() if pd.notna(k)}
    if actual_rows != EXPECTED_RAW_ROWS:
        raise AssertionError(f'Unexpected raw row count: {actual_rows}; expected {EXPECTED_RAW_ROWS}')
    if actual_counts != EXPECTED_RAW_LABEL_COUNTS:
        raise AssertionError(f'Unexpected raw label counts: {actual_counts}; expected {EXPECTED_RAW_LABEL_COUNTS}')


def estimate_tokens_from_text(text: str) -> int:
    # I use this lightweight estimate so the cost comparison stays reproducible offline.
    text = str(text)
    return max(1, math.ceil(len(text) / 4))


def estimate_tokens_for_series(texts: pd.Series) -> int:
    return int(texts.fillna('').astype(str).map(estimate_tokens_from_text).sum())


def estimate_embedding_cost_usd(model_name: str, total_tokens: int) -> float:
    price = EMBEDDING_PRICES_USD_PER_1M_TOKENS.get(model_name, 0.0)
    return round((total_tokens / 1_000_000) * price, 6)


def projected_seconds(seconds_per_email: float, email_count: int) -> float:
    return round(seconds_per_email * email_count, 4)


def projected_cost(cost_per_email: float, email_count: int) -> float:
    return round(cost_per_email * email_count, 6)


def add_cost_columns(row: dict[str, Any], row_count: int, token_cost_usd: float = 0.0, total_tokens: int = 0) -> dict[str, Any]:
    inference_seconds = float(row.get('inference_seconds', 0.0) or 0.0)
    feature_seconds = float(row.get('feature_generation_seconds', row.get('embedding_seconds', row.get('vectorizer_seconds', 0.0))) or 0.0)
    train_seconds = float(row.get('train_seconds', 0.0) or 0.0)
    total_runtime = feature_seconds + train_seconds + inference_seconds
    seconds_per_email = inference_seconds / row_count if row_count else 0.0
    cost_per_email = token_cost_usd / row_count if row_count else 0.0
    row.update({
        'total_runtime_seconds': round(total_runtime, 4),
        'inference_seconds_per_email': round(seconds_per_email, 8),
        'throughput_emails_per_second': round(row_count / inference_seconds, 4) if inference_seconds > 0 else np.nan,
        'estimated_input_tokens': int(total_tokens),
        'estimated_token_cost_usd': round(token_cost_usd, 6),
        'estimated_cost_per_email_usd': round(cost_per_email, 10),
        'projected_seconds_10k': projected_seconds(seconds_per_email, 10_000),
        'projected_seconds_100k': projected_seconds(seconds_per_email, 100_000),
        'projected_seconds_1m': projected_seconds(seconds_per_email, 1_000_000),
        'projected_cost_10k_usd': projected_cost(cost_per_email, 10_000),
        'projected_cost_100k_usd': projected_cost(cost_per_email, 100_000),
        'projected_cost_1m_usd': projected_cost(cost_per_email, 1_000_000),
    })
    return row


def classification_metrics(y_true: np.ndarray, y_pred: np.ndarray, scores: np.ndarray | None) -> dict[str, Any]:
    from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    metrics = {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
        'false_positive_rate': fp / (fp + tn) if (fp + tn) else 0.0,
        'false_negative_rate': fn / (fn + tp) if (fn + tp) else 0.0,
    }
    if scores is not None and len(np.unique(y_true)) == 2:
        try:
            metrics['roc_auc'] = roc_auc_score(y_true, scores)
        except ValueError:
            metrics['roc_auc'] = np.nan
    else:
        metrics['roc_auc'] = np.nan
    return metrics


def model_scores(model: Any, x: Any) -> np.ndarray | None:
    if hasattr(model, 'predict_proba'):
        proba = model.predict_proba(x)
        if proba.ndim == 2 and proba.shape[1] > 1:
            return proba[:, 1]
    if hasattr(model, 'decision_function'):
        return np.asarray(model.decision_function(x))
    return None


def evaluate_model(model: Any, x: Any, y: np.ndarray) -> tuple[dict[str, Any], np.ndarray, np.ndarray | None]:
    start = now_seconds()
    y_pred = model.predict(x)
    inference_seconds = elapsed(start)
    scores = model_scores(model, x)
    metrics = classification_metrics(y, y_pred, scores)
    metrics['inference_seconds'] = inference_seconds
    return metrics, np.asarray(y_pred), scores


def export_error_cases(df: pd.DataFrame, y_pred: np.ndarray, scores: np.ndarray | None, model_id: str) -> tuple[Path, Path]:
    ERROR_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    errors = df.copy()
    errors['predicted_label'] = y_pred
    errors['prediction_score'] = scores if scores is not None else np.nan
    cols = [c for c in ['source_dataset', 'subject', 'text', 'text_clean', 'label', 'predicted_label', 'prediction_score'] if c in errors.columns]
    false_negatives = errors[(errors['label'] == 1) & (errors['predicted_label'] == 0)]
    false_positives = errors[(errors['label'] == 0) & (errors['predicted_label'] == 1)]
    fn_path = ERROR_OUTPUT_DIR / f'{model_id}_false_negatives.csv'
    fp_path = ERROR_OUTPUT_DIR / f'{model_id}_false_positives.csv'
    false_negatives[cols].to_csv(fn_path, index=False)
    false_positives[cols].to_csv(fp_path, index=False)
    return fn_path, fp_path


ACADEMIC_BLUE = '#0072B2'
ACADEMIC_VERMILLION = '#D55E00'
ACADEMIC_GREEN = '#009E73'
ACADEMIC_PURPLE = '#CC79A7'
ACADEMIC_ORANGE = '#E69F00'
ACADEMIC_SKY_BLUE = '#56B4E9'
ACADEMIC_YELLOW = '#F0E442'
ACADEMIC_BLACK = '#222222'
ACADEMIC_GRAY = '#666666'
ACADEMIC_LIGHT_GRAY = '#D9D9D9'
ACADEMIC_PALETTE = [
    ACADEMIC_BLUE,
    ACADEMIC_VERMILLION,
    ACADEMIC_GREEN,
    ACADEMIC_PURPLE,
    ACADEMIC_ORANGE,
    ACADEMIC_SKY_BLUE,
    ACADEMIC_YELLOW,
    ACADEMIC_BLACK,
]
CLASS_COLORS = {0: ACADEMIC_BLUE, 1: ACADEMIC_VERMILLION}


def setup_academic_plotting() -> Any:
    import matplotlib
    matplotlib.use('Agg')
    from cycler import cycler
    from matplotlib import pyplot as plt

    plt.rcParams.update({
        'figure.dpi': 120,
        'savefig.dpi': 300,
        'font.size': 10,
        'axes.titlesize': 12,
        'axes.labelsize': 10,
        'legend.fontsize': 8.5,
        'xtick.labelsize': 9,
        'ytick.labelsize': 9,
        'axes.spines.top': False,
        'axes.spines.right': False,
        'axes.grid': False,
        'figure.facecolor': 'white',
        'axes.facecolor': 'white',
        'axes.edgecolor': ACADEMIC_BLACK,
        'axes.labelcolor': ACADEMIC_BLACK,
        'xtick.color': ACADEMIC_BLACK,
        'ytick.color': ACADEMIC_BLACK,
        'legend.frameon': False,
        'lines.linewidth': 1.8,
        'patch.edgecolor': 'white',
    })
    plt.rcParams['axes.prop_cycle'] = cycler(color=ACADEMIC_PALETTE)
    return plt


def style_academic_axis(ax: Any) -> None:
    ax.grid(False)
    ax.set_facecolor('white')
    ax.tick_params(axis='both', which='major', direction='out', length=4, width=0.8)
    for spine_name in ['left', 'bottom']:
        ax.spines[spine_name].set_color(ACADEMIC_BLACK)
        ax.spines[spine_name].set_linewidth(0.8)
    if 'top' in ax.spines:
        ax.spines['top'].set_visible(False)
    if 'right' in ax.spines:
        ax.spines['right'].set_visible(False)


def rotate_category_labels(ax: Any, rotation: int = 45) -> None:
    ax.tick_params(axis='x', rotation=rotation)
    for label in ax.get_xticklabels():
        label.set_horizontalalignment('right')
        label.set_rotation_mode('anchor')


def make_transparent_legend(ax: Any, *args: Any, **kwargs: Any) -> Any:
    kwargs.setdefault('frameon', True)
    kwargs.setdefault('facecolor', 'white')
    kwargs.setdefault('edgecolor', ACADEMIC_LIGHT_GRAY)
    kwargs.setdefault('framealpha', 0.68)
    legend = ax.legend(*args, **kwargs)
    if legend is not None:
        legend.get_frame().set_linewidth(0.8)
    return legend


def save_academic_figure(fig: Any, path_stem: Path) -> None:
    path_stem.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    # I only save PNG files because these are easier to inspect directly in PyCharm.
    fig.savefig(path_stem.with_suffix('.png'), dpi=300, bbox_inches='tight')


def save_confusion_matrix_figure(metrics: dict[str, Any], title: str, path_stem: Path) -> None:
    plt = setup_academic_plotting()
    matrix = np.array([[metrics['tn'], metrics['fp']], [metrics['fn'], metrics['tp']]])
    fig, ax = plt.subplots(figsize=(4.8, 4.2))
    image = ax.imshow(matrix, cmap='Blues', interpolation='nearest', aspect='equal')

    ax.set_xticks([0, 1], labels=['Predicted legitimate', 'Predicted phishing'])
    ax.set_yticks([0, 1], labels=['Actual legitimate', 'Actual phishing'])
    ax.set_xlim(-0.5, 1.5)
    ax.set_ylim(1.5, -0.5)
    ax.set_title(title)

    ax.grid(False, which='both', axis='both')
    ax.minorticks_off()
    ax.tick_params(axis='both', which='both', length=0)
    ax.set_xticks([], minor=True)
    ax.set_yticks([], minor=True)

    threshold = matrix.max() / 2 if matrix.size else 0
    for i in range(2):
        for j in range(2):
            text_color = 'white' if matrix[i, j] > threshold else 'black'
            ax.text(j, i, str(matrix[i, j]), ha='center', va='center', color=text_color)

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color(ACADEMIC_BLACK)
        spine.set_linewidth(0.8)

    colorbar = fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    colorbar.ax.grid(False, which='both')
    colorbar.ax.minorticks_off()

    save_academic_figure(fig, path_stem)
    plt.close(fig)


def artifact_size(path: Path) -> int:
    return int(path.stat().st_size) if path.exists() else 0


ensure_output_dirs()
print('Notebook configuration loaded.')
print('Code directory:', CODE_DIR)
print('Output directory:', OUTPUT_DIR)


Notebook configuration loaded.
Code directory: /Users/tiencains/Documents/dissertation/dissertation process/code/phishing_email_detection
Output directory: /Users/tiencains/Documents/dissertation/dissertation process/code/phishing_email_detection/output


In [2]:
def stratified_split(df: pd.DataFrame, label_col: str, test_size: float, random_seed: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    from sklearn.model_selection import train_test_split
    train_df, test_df = train_test_split(df, test_size=test_size, stratify=df[label_col], random_state=random_seed)
    return train_df.reset_index(drop=True), test_df.reset_index(drop=True)


def make_balanced_training_set(train_df: pd.DataFrame, random_seed: int) -> pd.DataFrame:
    counts = train_df['label'].value_counts()
    min_count = int(counts.min())
    parts = [train_df[train_df['label'] == label].sample(n=min_count, random_state=random_seed) for label in sorted(counts.index)]
    return pd.concat(parts, ignore_index=True).sample(frac=1.0, random_state=random_seed).reset_index(drop=True)


def run_preprocess_and_split() -> dict[str, pd.DataFrame]:
    ensure_output_dirs()
    require_packages(['sklearn', 'matplotlib'])
    plt = setup_academic_plotting()

    input_path = DATA_OUTPUT_DIR / 'merged_ceas_enron_raw.csv'
    if not input_path.exists():
        raise FileNotFoundError('Run the merge and describe notebook before preprocessing.')

    df = pd.read_csv(input_path, low_memory=False)
    df['label'] = pd.to_numeric(df['label'], errors='coerce').astype(int)
    df['text'] = df['text'].fillna('').astype(str)
    df['text_clean'] = df['text'].map(clean_text_level1)
    df['dedup_key'] = make_dedup_key(df['text'])

    before_empty = len(df)
    df = df[df['text_clean'].str.len() > 0].copy()
    empty_removed = before_empty - len(df)

    conflict_counts = df.groupby('dedup_key')['label'].nunique()
    conflicting_texts = conflict_counts[conflict_counts > 1].index
    conflicting_rows = df[df['dedup_key'].isin(conflicting_texts)].copy()
    conflicting_rows.to_csv(DATA_OUTPUT_DIR / 'conflicting_duplicate_texts.csv', index=False)
    df = df[~df['dedup_key'].isin(conflicting_texts)].copy()

    before_dedup = len(df)
    df = df.drop_duplicates(subset=['dedup_key'], keep='first').reset_index(drop=True)
    duplicate_removed = before_dedup - len(df)
    df = df.drop(columns=['dedup_key'])

    processed_path = DATA_OUTPUT_DIR / 'processed_deduplicated.csv'
    df.to_csv(processed_path, index=False)

    train_val, test_df = stratified_split(df, 'label', test_size=.20, random_seed=RANDOM_SEED)
    train_df, validation_df = stratified_split(train_val, 'label', test_size=.25, random_seed=RANDOM_SEED)
    train_balanced_df = make_balanced_training_set(train_df, RANDOM_SEED)

    train_df.to_csv(DATA_OUTPUT_DIR / 'train_unbalanced.csv', index=False)
    train_balanced_df.to_csv(DATA_OUTPUT_DIR / 'train_balanced.csv', index=False)
    validation_df.to_csv(DATA_OUTPUT_DIR / 'validation.csv', index=False)
    test_df.to_csv(DATA_OUTPUT_DIR / 'test_imbalanced.csv', index=False)

    split_rows = []
    for name, split_df in [('processed_deduplicated', df), ('train_unbalanced', train_df), ('train_balanced', train_balanced_df), ('validation', validation_df), ('test_imbalanced', test_df)]:
        counts = split_df['label'].value_counts().to_dict()
        split_rows.append({
            'split': name,
            'rows': len(split_df),
            'label_0_legitimate': int(counts.get(0, 0)),
            'label_1_phishing_spam': int(counts.get(1, 0)),
            'label_1_rate': float((split_df['label'] == 1).mean()),
        })
    split_summary = pd.DataFrame(split_rows)
    split_summary.to_csv(DATA_OUTPUT_DIR / 'split_summary.csv', index=False)

    quality = pd.DataFrame([{
        'empty_text_rows_removed': int(empty_removed),
        'conflicting_duplicate_rows_removed': int(len(conflicting_rows)),
        'deduplicate_rows_removed': int(duplicate_removed),
        'final_processed_rows': int(len(df)),
        'train_balanced_label_counts_equal': bool(train_balanced_df['label'].value_counts().nunique() == 1),
        'test_label_1_rate': float((test_df['label'] == 1).mean()),
    }])
    quality.to_csv(DATA_OUTPUT_DIR / 'preprocessing_quality_checks.csv', index=False)

    if train_balanced_df['label'].value_counts().nunique() != 1:
        raise AssertionError('Balanced training set does not have equal label counts.')
    if int((df['text_clean'].str.len() == 0).sum()) != 0:
        raise AssertionError('Empty cleaned texts remain after preprocessing.')

    fig, ax = plt.subplots(figsize=(7.2, 4.5))
    split_plot = split_summary.set_index('split')[['label_0_legitimate', 'label_1_phishing_spam']]
    split_plot.plot(kind='bar', stacked=True, ax=ax, color=[ACADEMIC_BLUE, ACADEMIC_VERMILLION])
    ax.set_title('Class Counts Across Data Splits')
    ax.set_xlabel('Data split')
    ax.set_ylabel('Email count')
    make_transparent_legend(ax, ['Legitimate (0)', 'Phishing/spam (1)'], loc='upper right')
    rotate_category_labels(ax)
    save_academic_figure(fig, FIGURE_OUTPUT_DIR / 'split_class_counts')
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(6.5, 4.0))
    removal = pd.Series({'Empty text': empty_removed, 'Conflicting duplicates': len(conflicting_rows), 'Duplicate texts': duplicate_removed})
    ax.bar(removal.index, removal.values, color=ACADEMIC_VERMILLION)
    ax.set_title('Rows Removed During Preprocessing')
    ax.set_ylabel('Rows removed')
    rotate_category_labels(ax)
    save_academic_figure(fig, FIGURE_OUTPUT_DIR / 'preprocessing_rows_removed')
    plt.close(fig)

    print(f'Processed data saved to {processed_path}')
    display(split_summary)
    return {'train_balanced': train_balanced_df, 'validation': validation_df, 'test_imbalanced': test_df}


In [3]:
def load_split(filename: str) -> pd.DataFrame:
    df = pd.read_csv(DATA_OUTPUT_DIR / filename)
    df['label'] = pd.to_numeric(df['label'], errors='coerce').astype(int)
    if 'text_clean' not in df.columns:
        df['text_clean'] = df['text'].fillna('').astype(str).map(clean_text_level1)
    return df[df['text_clean'].fillna('').astype(str).str.len() > 0].reset_index(drop=True)


def sample_split(df: pd.DataFrame, rows_per_label: int, random_seed: int) -> pd.DataFrame:
    parts = []
    for label, group in df.groupby('label'):
        parts.append(group.sample(n=min(rows_per_label, len(group)), random_state=random_seed))
    return pd.concat(parts, ignore_index=True).sample(frac=1.0, random_state=random_seed).reset_index(drop=True)


def get_classifiers(feature_type: str, random_seed: int, rf_trees: int) -> dict[str, Any]:
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.linear_model import LogisticRegression
    from sklearn.naive_bayes import GaussianNB, MultinomialNB
    from sklearn.svm import LinearSVC
    nb_model = MultinomialNB() if feature_type == 'tfidf' else GaussianNB()
    return {
        'lr': LogisticRegression(max_iter=2000, class_weight='balanced', random_state=random_seed, n_jobs=1),
        'nb': nb_model,
        'rf': RandomForestClassifier(n_estimators=rf_trees, random_state=random_seed, max_features='sqrt', class_weight='balanced_subsample', n_jobs=1),
        'svm': LinearSVC(C=1.0, class_weight='balanced', random_state=random_seed, max_iter=5000),
    }


def average_word_vectors(model: Any, token_lists: list[list[str]], vector_size: int) -> np.ndarray:
    vectors = np.zeros((len(token_lists), vector_size), dtype=np.float32)
    keyed_vectors = model.wv
    for i, tokens in enumerate(token_lists):
        present = [keyed_vectors[token] for token in tokens if token in keyed_vectors]
        if present:
            vectors[i] = np.mean(present, axis=0)
    return vectors


def run_ml_baselines(run_quick: bool = False, quick_rows_per_label: int = 1200, max_features: int = 20000, word2vec_size: int = 100, rf_trees: int = 100) -> pd.DataFrame:
    ensure_output_dirs()
    require_packages(['sklearn', 'gensim', 'joblib', 'matplotlib'])
    import joblib
    from gensim.models import Word2Vec
    from sklearn.feature_extraction.text import TfidfVectorizer
    plt = setup_academic_plotting()

    train_df = load_split('train_balanced.csv')
    validation_df = load_split('validation.csv')
    test_df = load_split('test_imbalanced.csv')

    if run_quick:
        max_features = min(max_features, 3000)
        word2vec_size = min(word2vec_size, 50)
        rf_trees = min(rf_trees, 25)
        train_df = sample_split(train_df, quick_rows_per_label, RANDOM_SEED)
        validation_df = sample_split(validation_df, max(200, quick_rows_per_label // 4), RANDOM_SEED)
        test_df = sample_split(test_df, max(200, quick_rows_per_label // 4), RANDOM_SEED)
        print('Running quick smoke test with sampled data.')

    y_train = train_df['label'].to_numpy()
    y_validation = validation_df['label'].to_numpy()
    y_test = test_df['label'].to_numpy()
    results: list[dict[str, Any]] = []

    start = now_seconds()
    tfidf = TfidfVectorizer(max_features=max_features, ngram_range=(1, 2), min_df=2, max_df=.95, sublinear_tf=True)
    x_train_tfidf = tfidf.fit_transform(train_df['text_clean'])
    x_validation_tfidf = tfidf.transform(validation_df['text_clean'])
    x_test_tfidf = tfidf.transform(test_df['text_clean'])
    tfidf_seconds = elapsed(start)

    for model_name, classifier in get_classifiers('tfidf', RANDOM_SEED, rf_trees).items():
        model_id = f'tfidf_{model_name}'
        print(f'Training {model_id}...')
        start = now_seconds()
        classifier.fit(x_train_tfidf, y_train)
        train_seconds = elapsed(start)
        artifact_path = MODEL_OUTPUT_DIR / f'{model_id}.joblib'
        joblib.dump({'feature_type': 'tfidf', 'model_name': model_name, 'vectorizer': tfidf, 'classifier': classifier, 'text_column': 'text_clean'}, artifact_path)
        for split_name, x_split, y_split, row_count in [('validation', x_validation_tfidf, y_validation, len(validation_df)), ('test', x_test_tfidf, y_test, len(test_df))]:
            metrics, _, _ = evaluate_model(classifier, x_split, y_split)
            row = {
                'feature_type': 'tfidf', 'model_name': model_name, 'model_id': model_id, 'evaluation_split': split_name,
                'train_rows': len(train_df), 'validation_rows': len(validation_df), 'test_rows': len(test_df),
                'feature_generation_seconds': tfidf_seconds, 'train_seconds': train_seconds, 'artifact_path': str(artifact_path),
                'model_artifact_size_bytes': artifact_size(artifact_path), 'deployment_type': 'local', 'api_required': False,
                'local_only': True, 'runtime_dependency': 'scikit-learn', 'token_cost_method': 'not_applicable',
            }
            row.update(metrics)
            results.append(add_cost_columns(row, row_count))
            if split_name == 'test':
                save_confusion_matrix_figure(metrics, f'{model_id} Confusion Matrix', FIGURE_OUTPUT_DIR / f'confusion_matrix_{model_id}')

    train_tokens = [simple_tokenize(text) for text in train_df['text_clean']]
    validation_tokens = [simple_tokenize(text) for text in validation_df['text_clean']]
    test_tokens = [simple_tokenize(text) for text in test_df['text_clean']]
    start = now_seconds()
    word2vec = Word2Vec(sentences=train_tokens, vector_size=word2vec_size, window=5, min_count=2, workers=1, sg=1, seed=RANDOM_SEED, epochs=10)
    x_train_w2v = average_word_vectors(word2vec, train_tokens, word2vec_size)
    x_validation_w2v = average_word_vectors(word2vec, validation_tokens, word2vec_size)
    x_test_w2v = average_word_vectors(word2vec, test_tokens, word2vec_size)
    word2vec_seconds = elapsed(start)

    for model_name, classifier in get_classifiers('word2vec', RANDOM_SEED, rf_trees).items():
        model_id = f'word2vec_{model_name}'
        print(f'Training {model_id}...')
        start = now_seconds()
        classifier.fit(x_train_w2v, y_train)
        train_seconds = elapsed(start)
        artifact_path = MODEL_OUTPUT_DIR / f'{model_id}.joblib'
        joblib.dump({'feature_type': 'word2vec', 'model_name': model_name, 'word2vec': word2vec, 'classifier': classifier, 'word2vec_size': word2vec_size, 'text_column': 'text_clean'}, artifact_path)
        for split_name, x_split, y_split, row_count in [('validation', x_validation_w2v, y_validation, len(validation_df)), ('test', x_test_w2v, y_test, len(test_df))]:
            metrics, _, _ = evaluate_model(classifier, x_split, y_split)
            row = {
                'feature_type': 'word2vec', 'model_name': model_name, 'model_id': model_id, 'evaluation_split': split_name,
                'train_rows': len(train_df), 'validation_rows': len(validation_df), 'test_rows': len(test_df),
                'feature_generation_seconds': word2vec_seconds, 'train_seconds': train_seconds, 'artifact_path': str(artifact_path),
                'model_artifact_size_bytes': artifact_size(artifact_path), 'deployment_type': 'local', 'api_required': False,
                'local_only': True, 'runtime_dependency': 'gensim + scikit-learn', 'token_cost_method': 'not_applicable',
            }
            row.update(metrics)
            results.append(add_cost_columns(row, row_count))
            if split_name == 'test':
                save_confusion_matrix_figure(metrics, f'{model_id} Confusion Matrix', FIGURE_OUTPUT_DIR / f'confusion_matrix_{model_id}')

    results_df = pd.DataFrame(results)
    results_path = REPORT_OUTPUT_DIR / 'ml_baseline_results.csv'
    results_df.to_csv(results_path, index=False)
    champion = results_df[results_df['evaluation_split'] == 'validation'].sort_values(['f1', 'balanced_accuracy'], ascending=False).head(1)
    champion.to_csv(REPORT_OUTPUT_DIR / 'ml_champion.csv', index=False)

    metric_cols = ['accuracy', 'precision', 'recall', 'f1', 'balanced_accuracy', 'roc_auc']
    test_results = results_df[results_df['evaluation_split'] == 'test'].copy()
    fig, ax = plt.subplots(figsize=(9.0, 5.0))
    test_results.set_index('model_id')[metric_cols].plot(kind='bar', ax=ax)
    ax.set_title('Traditional ML Test Performance')
    ax.set_ylabel('Score')
    ax.set_ylim(0, 1.05)
    rotate_category_labels(ax)
    make_transparent_legend(ax, loc='lower right', ncol=2)
    save_academic_figure(fig, FIGURE_OUTPUT_DIR / 'ml_model_metric_comparison')
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(7.5, 4.5))
    ax.bar(test_results['model_id'], test_results['total_runtime_seconds'], color=ACADEMIC_SKY_BLUE)
    ax.set_title('Traditional ML Runtime Comparison')
    ax.set_ylabel('Total runtime seconds')
    rotate_category_labels(ax)
    save_academic_figure(fig, FIGURE_OUTPUT_DIR / 'ml_runtime_comparison')
    plt.close(fig)

    print(f'ML baseline results saved to {results_path}')
    display(results_df)
    return results_df


In [4]:
def load_embedding_split(filename: str) -> pd.DataFrame:
    df = pd.read_csv(DATA_OUTPUT_DIR / filename)
    df['label'] = pd.to_numeric(df['label'], errors='coerce').astype(int)
    if 'text_clean' not in df.columns:
        df['text_clean'] = df['text'].fillna('').astype(str).map(clean_text_level1)
    return df[df['text_clean'].fillna('').astype(str).str.len() > 0].reset_index(drop=True)


def embed_texts(client: Any, texts: list[str], model: str, batch_size: int) -> np.ndarray:
    vectors: list[list[float]] = []
    for start_idx in range(0, len(texts), batch_size):
        batch = texts[start_idx:start_idx + batch_size]
        response = client.embeddings.create(model=model, input=batch)
        vectors.extend([item.embedding for item in response.data])
        print(f'Embedded {min(start_idx + batch_size, len(texts))}/{len(texts)} rows with {model}')
    return np.asarray(vectors, dtype=np.float32)


def load_or_create_embeddings(client: Any, df: pd.DataFrame, split_name: str, embedding_model: str, batch_size: int, max_chars: int, quick_suffix: str = '') -> tuple[np.ndarray, float, int, float]:
    safe_model_name = embedding_model.replace('/', '_')
    cache_path = EMBEDDING_OUTPUT_DIR / f'{safe_model_name}_{split_name}{quick_suffix}.npy'
    token_path = EMBEDDING_OUTPUT_DIR / f'{safe_model_name}_{split_name}{quick_suffix}_usage.json'
    total_tokens = estimate_tokens_for_series(df['text_clean'].str.slice(0, max_chars))
    estimated_cost = estimate_embedding_cost_usd(embedding_model, total_tokens)
    if cache_path.exists():
        vectors = np.load(cache_path)
        if len(vectors) == len(df):
            return vectors, 0.0, total_tokens, estimated_cost
    start = now_seconds()
    vectors = embed_texts(client, df['text_clean'].str.slice(0, max_chars).tolist(), embedding_model, batch_size)
    embedding_seconds = elapsed(start)
    np.save(cache_path, vectors)
    save_json({'split': split_name, 'embedding_model': embedding_model, 'estimated_tokens': total_tokens, 'estimated_cost_usd': estimated_cost}, token_path)
    return vectors, embedding_seconds, total_tokens, estimated_cost


def get_embedding_classifiers(random_seed: int, rf_trees: int) -> dict[str, Any]:
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.linear_model import LogisticRegression
    from sklearn.svm import LinearSVC
    return {
        'lr': LogisticRegression(max_iter=2000, class_weight='balanced', random_state=random_seed, n_jobs=1),
        'svm': LinearSVC(C=1.0, class_weight='balanced', random_state=random_seed, max_iter=5000),
        'rf': RandomForestClassifier(n_estimators=rf_trees, random_state=random_seed, max_features='sqrt', class_weight='balanced_subsample', n_jobs=1),
    }


def run_openai_embedding_experiments(run_quick: bool = False, quick_rows_per_label: int = 150, embedding_models: list[str] | None = None, batch_size: int = 100, max_chars: int = 6000, rf_trees: int = 100) -> pd.DataFrame:
    ensure_output_dirs()
    require_packages(['sklearn', 'joblib', 'openai'])
    import joblib
    from openai import OpenAI

    api_key = load_openai_api_key()
    if not api_key:
        reason = f'Embedding API key was not found. I need to add it to {API_KEY_FILE}. Optional embedding experiments skipped.'
        (REPORT_OUTPUT_DIR / 'openai_embedding_status.txt').write_text(reason + '\n', encoding='utf-8')
        empty_columns = [
            'feature_type', 'embedding_model', 'model_name', 'model_id', 'evaluation_split',
            'accuracy', 'precision', 'recall', 'f1', 'balanced_accuracy', 'roc_auc',
            'deployment_type', 'api_required', 'estimated_input_tokens', 'estimated_token_cost_usd'
        ]
        empty = pd.DataFrame(columns=empty_columns)
        empty.to_csv(REPORT_OUTPUT_DIR / 'openai_embedding_results.csv', index=False)
        print(reason)
        return empty

    embedding_models = embedding_models or ['text-embedding-3-small', 'text-embedding-3-large']
    train_df = load_embedding_split('train_balanced.csv')
    validation_df = load_embedding_split('validation.csv')
    test_df = load_embedding_split('test_imbalanced.csv')

    if run_quick:
        rf_trees = min(rf_trees, 25)
        train_df = sample_split(train_df, quick_rows_per_label, RANDOM_SEED)
        validation_df = sample_split(validation_df, max(50, quick_rows_per_label // 3), RANDOM_SEED)
        test_df = sample_split(test_df, max(50, quick_rows_per_label // 3), RANDOM_SEED)
        print('Running quick embedding test with sampled data.')

    client = OpenAI(api_key=api_key)
    all_results: list[dict[str, Any]] = []
    usage_rows: list[dict[str, Any]] = []

    for embedding_model in embedding_models:
        quick_suffix = '_quick' if run_quick else ''
        x_train, train_embed_seconds, train_tokens, train_cost = load_or_create_embeddings(client, train_df, 'train_balanced', embedding_model, batch_size, max_chars, quick_suffix)
        x_validation, validation_embed_seconds, validation_tokens, validation_cost = load_or_create_embeddings(client, validation_df, 'validation', embedding_model, batch_size, max_chars, quick_suffix)
        x_test, test_embed_seconds, test_tokens, test_cost = load_or_create_embeddings(client, test_df, 'test_imbalanced', embedding_model, batch_size, max_chars, quick_suffix)
        usage_rows.extend([
            {'embedding_model': embedding_model, 'split': 'train_balanced', 'estimated_tokens': train_tokens, 'estimated_cost_usd': train_cost, 'embedding_seconds': train_embed_seconds},
            {'embedding_model': embedding_model, 'split': 'validation', 'estimated_tokens': validation_tokens, 'estimated_cost_usd': validation_cost, 'embedding_seconds': validation_embed_seconds},
            {'embedding_model': embedding_model, 'split': 'test_imbalanced', 'estimated_tokens': test_tokens, 'estimated_cost_usd': test_cost, 'embedding_seconds': test_embed_seconds},
        ])

        y_train = train_df['label'].to_numpy()
        for model_name, classifier in get_embedding_classifiers(RANDOM_SEED, rf_trees).items():
            safe_model = embedding_model.replace('-', '_').replace('/', '_')
            model_id = f'{safe_model}_{model_name}'
            print(f'Training {model_id}...')
            start = now_seconds()
            classifier.fit(x_train, y_train)
            train_seconds = elapsed(start)
            artifact_path = MODEL_OUTPUT_DIR / f'openai_embedding_{model_id}.joblib'
            joblib.dump({'feature_type': 'openai_embedding', 'embedding_model': embedding_model, 'model_name': model_name, 'classifier': classifier, 'max_chars': max_chars, 'text_column': 'text_clean'}, artifact_path)

            for split_name, x_split, y_split, row_count, embed_seconds, tokens, cost in [
                ('validation', x_validation, validation_df['label'].to_numpy(), len(validation_df), validation_embed_seconds, validation_tokens, validation_cost),
                ('test', x_test, test_df['label'].to_numpy(), len(test_df), test_embed_seconds, test_tokens, test_cost),
            ]:
                metrics, _, _ = evaluate_model(classifier, x_split, y_split)
                row = {
                    'feature_type': 'openai_embedding', 'embedding_model': embedding_model, 'model_name': model_name,
                    'model_id': model_id, 'evaluation_split': split_name, 'train_rows': len(train_df),
                    'validation_rows': len(validation_df), 'test_rows': len(test_df), 'feature_generation_seconds': embed_seconds,
                    'embedding_seconds': embed_seconds, 'train_seconds': train_seconds, 'artifact_path': str(artifact_path),
                    'model_artifact_size_bytes': artifact_size(artifact_path), 'deployment_type': 'api_embedding_plus_local_classifier',
                    'api_required': True, 'local_only': False, 'runtime_dependency': 'openai + scikit-learn',
                    'token_cost_method': 'estimated_from_character_count',
                }
                row.update(metrics)
                all_results.append(add_cost_columns(row, row_count, token_cost_usd=cost, total_tokens=tokens))

    results_df = pd.DataFrame(all_results)
    results_df.to_csv(REPORT_OUTPUT_DIR / 'openai_embedding_results.csv', index=False)
    pd.DataFrame(usage_rows).to_csv(REPORT_OUTPUT_DIR / 'openai_embedding_usage_estimates.csv', index=False)
    if len(results_df):
        champion = results_df[results_df['evaluation_split'] == 'validation'].sort_values(['f1', 'balanced_accuracy'], ascending=False).head(1)
        champion.to_csv(REPORT_OUTPUT_DIR / 'embedding_champion.csv', index=False)
    (REPORT_OUTPUT_DIR / 'openai_embedding_status.txt').write_text('OpenAI embedding experiments completed.\n', encoding='utf-8')
    print(f'OpenAI embedding results saved to {REPORT_OUTPUT_DIR / "openai_embedding_results.csv"}')
    display(results_df)
    return results_df


In [5]:
embedding_results = run_openai_embedding_experiments(run_quick=False)


Embedded 100/39314 rows with text-embedding-3-small
Embedded 200/39314 rows with text-embedding-3-small
Embedded 300/39314 rows with text-embedding-3-small
Embedded 400/39314 rows with text-embedding-3-small
Embedded 500/39314 rows with text-embedding-3-small
Embedded 600/39314 rows with text-embedding-3-small
Embedded 700/39314 rows with text-embedding-3-small
Embedded 800/39314 rows with text-embedding-3-small
Embedded 900/39314 rows with text-embedding-3-small
Embedded 1000/39314 rows with text-embedding-3-small
Embedded 1100/39314 rows with text-embedding-3-small
Embedded 1200/39314 rows with text-embedding-3-small
Embedded 1300/39314 rows with text-embedding-3-small
Embedded 1400/39314 rows with text-embedding-3-small
Embedded 1500/39314 rows with text-embedding-3-small
Embedded 1600/39314 rows with text-embedding-3-small
Embedded 1700/39314 rows with text-embedding-3-small
Embedded 1800/39314 rows with text-embedding-3-small
Embedded 1900/39314 rows with text-embedding-3-small
Em

,feature_type,embedding_model,model_name,model_id,evaluation_split,train_rows,validation_rows,test_rows,feature_generation_seconds,embedding_seconds,...,throughput_emails_per_second,estimated_input_tokens,estimated_token_cost_usd,estimated_cost_per_email_usd,projected_seconds_10k,projected_seconds_100k,projected_seconds_1m,projected_cost_10k_usd,projected_cost_100k_usd,projected_cost_1m_usd
0,openai_embedding,text-embedding-3-small,lr,text_embedding_3_small_lr,validation,39314,13677,13677,68.7485,68.7485,...,146906.5521,4117249,0.082345,0.000006,0.0681,0.6807,6.8070,0.060207,0.602069,6.020692
1,openai_embedding,text-embedding-3-small,lr,text_embedding_3_small_lr,test,39314,13677,13677,63.8571,63.8571,...,743315.2174,4002573,0.080051,0.000006,0.0135,0.1345,1.3453,0.058530,0.585296,5.852965
2,openai_embedding,text-embedding-3-small,svm,text_embedding_3_small_svm,validation,39314,13677,13677,68.7485,68.7485,...,217440.3816,4117249,0.082345,0.000006,0.0460,0.4599,4.5990,0.060207,0.602069,6.020692
3,openai_embedding,text-embedding-3-small,svm,text_embedding_3_small_svm,test,39314,13677,13677,63.8571,63.8571,...,499160.5839,4002573,0.080051,0.000006,0.0200,0.2003,2.0034,0.058530,0.585296,5.852965
4,openai_embedding,text-embedding-3-small,rf,text_embedding_3_small_rf,validation,39314,13677,13677,68.7485,68.7485,...,82540.7363,4117249,0.082345,0.000006,0.1212,1.2115,12.1152,0.060207,0.602069,6.020692
5,openai_embedding,text-embedding-3-small,rf,text_embedding_3_small_rf,test,39314,13677,13677,63.8571,63.8571,...,78333.3333,4002573,0.080051,0.000006,0.1277,1.2766,12.7660,0.058530,0.585296,5.852965
6,openai_embedding,text-embedding-3-large,lr,text_embedding_3_large_lr,validation,39314,13677,13677,90.3300,90.3300,...,55824.4898,4117249,0.535242,0.000039,0.1791,1.7913,17.9133,0.391345,3.913446,39.134459
7,openai_embedding,text-embedding-3-large,lr,text_embedding_3_large_lr,test,39314,13677,13677,90.4671,90.4671,...,893921.5686,4002573,0.520334,0.000038,0.0112,0.1119,1.1187,0.380445,3.804445,38.044454
8,openai_embedding,text-embedding-3-large,svm,text_embedding_3_large_svm,validation,39314,13677,13677,90.3300,90.3300,...,111014.6104,4117249,0.535242,0.000039,0.0901,0.9008,9.0078,0.391345,3.913446,39.134459
9,openai_embedding,text-embedding-3-large,svm,text_embedding_3_large_svm,test,39314,13677,13677,90.4671,90.4671,...,180673.7120,4002573,0.520334,0.000038,0.0553,0.5535,5.5348,0.380445,3.804445,38.044454


## Preview generated embedding result tables


In [2]:
from IPython.display import Image, display


def show_csv_head(path, n=5):
    path = Path(path)
    print(f'CSV preview: {path}')
    if path.exists():
        try:
            display(pd.read_csv(path).head(n))
        except pd.errors.EmptyDataError:
            print('CSV exists but is empty because the optional stage was skipped.')
    else:
        print('Not found yet. Run the earlier cells first.')


def show_png(path):
    path = Path(path)
    print(f'PNG preview: {path}')
    if path.exists():
        display(Image(filename=str(path)))
    else:
        print('Not found yet. Run the earlier cells first.')

show_csv_head(REPORT_OUTPUT_DIR / 'openai_embedding_results.csv')
show_csv_head(REPORT_OUTPUT_DIR / 'openai_embedding_usage_estimates.csv')
show_csv_head(REPORT_OUTPUT_DIR / 'embedding_champion.csv')
status_path = REPORT_OUTPUT_DIR / 'openai_embedding_status.txt'
print('Embedding status:')
print(status_path.read_text(encoding='utf-8') if status_path.exists() else 'Not found yet.')


CSV preview: /Users/tiencains/Documents/dissertation/dissertation process/code/phishing_email_detection/output/reports/openai_embedding_results.csv


,feature_type,embedding_model,model_name,model_id,evaluation_split,train_rows,validation_rows,test_rows,feature_generation_seconds,embedding_seconds,...,throughput_emails_per_second,estimated_input_tokens,estimated_token_cost_usd,estimated_cost_per_email_usd,projected_seconds_10k,projected_seconds_100k,projected_seconds_1m,projected_cost_10k_usd,projected_cost_100k_usd,projected_cost_1m_usd
0,openai_embedding,text-embedding-3-small,lr,text_embedding_3_small_lr,validation,39314,13677,13677,68.7485,68.7485,...,146906.5521,4117249,0.082345,0.000006,0.0681,0.6807,6.8070,0.060207,0.602069,6.020692
1,openai_embedding,text-embedding-3-small,lr,text_embedding_3_small_lr,test,39314,13677,13677,63.8571,63.8571,...,743315.2174,4002573,0.080051,0.000006,0.0135,0.1345,1.3453,0.058530,0.585296,5.852965
2,openai_embedding,text-embedding-3-small,svm,text_embedding_3_small_svm,validation,39314,13677,13677,68.7485,68.7485,...,217440.3816,4117249,0.082345,0.000006,0.0460,0.4599,4.5990,0.060207,0.602069,6.020692
3,openai_embedding,text-embedding-3-small,svm,text_embedding_3_small_svm,test,39314,13677,13677,63.8571,63.8571,...,499160.5839,4002573,0.080051,0.000006,0.0200,0.2003,2.0034,0.058530,0.585296,5.852965
4,openai_embedding,text-embedding-3-small,rf,text_embedding_3_small_rf,validation,39314,13677,13677,68.7485,68.7485,...,82540.7363,4117249,0.082345,0.000006,0.1212,1.2115,12.1152,0.060207,0.602069,6.020692


CSV preview: /Users/tiencains/Documents/dissertation/dissertation process/code/phishing_email_detection/output/reports/openai_embedding_usage_estimates.csv


,embedding_model,split,estimated_tokens,estimated_cost_usd,embedding_seconds
0,text-embedding-3-small,train_balanced,11728877,0.234578,191.1220
1,text-embedding-3-small,validation,4117249,0.082345,68.7485
2,text-embedding-3-small,test_imbalanced,4002573,0.080051,63.8571
3,text-embedding-3-large,train_balanced,11728877,1.524754,383.2362
4,text-embedding-3-large,validation,4117249,0.535242,90.3300


CSV preview: /Users/tiencains/Documents/dissertation/dissertation process/code/phishing_email_detection/output/reports/embedding_champion.csv


,feature_type,embedding_model,model_name,model_id,evaluation_split,train_rows,validation_rows,test_rows,feature_generation_seconds,embedding_seconds,...,throughput_emails_per_second,estimated_input_tokens,estimated_token_cost_usd,estimated_cost_per_email_usd,projected_seconds_10k,projected_seconds_100k,projected_seconds_1m,projected_cost_10k_usd,projected_cost_100k_usd,projected_cost_1m_usd
0,openai_embedding,text-embedding-3-large,svm,text_embedding_3_large_svm,validation,39314,13677,13677,90.33,90.33,...,111014.6104,4117249,0.535242,0.000039,0.0901,0.9008,9.0078,0.391345,3.913446,39.134459


Embedding status:
OpenAI embedding experiments completed.

